# 03 · Streaming inference

Structured Streaming: as new frame/track files land, infer per micro-batch and
append to a Delta sink. Uses Auto Loader (`cloudFiles`).

In [ ]:
# ---- CONFIG (the only cell you must edit) --------------------------------
dbutils.widgets.text("input_delta_path", "drone.tracks", "Input Delta table/path")
dbutils.widgets.text("output_delta_path", "drone.inference_results", "Output Delta table/path")
dbutils.widgets.text("model_name", "visdrone-yolov8x", "DVSA model name")
dbutils.widgets.text("batch_size", "64", "Batch size")
dbutils.widgets.text("checkpoint_location", "/Volumes/drone/_chk", "Checkpoint (streaming)")
dbutils.widgets.text("secret_scope", "dvsa", "Databricks Secret scope")

INPUT_DELTA = dbutils.widgets.get("input_delta_path")
OUTPUT_DELTA = dbutils.widgets.get("output_delta_path")
MODEL_NAME = dbutils.widgets.get("model_name")
BATCH_SIZE = int(dbutils.widgets.get("batch_size"))
CHECKPOINT = dbutils.widgets.get("checkpoint_location")
SECRET_SCOPE = dbutils.widgets.get("secret_scope")

In [ ]:
from dvsa_databricks_connector import config_from_secrets, stream_inference

cfg = config_from_secrets(dbutils, scope=SECRET_SCOPE)
INPUT_STREAM_PATH = "/Volumes/drone/landing/frames"

query = stream_inference(
    spark,
    input_path=INPUT_STREAM_PATH,
    model_name=MODEL_NAME,
    checkpoint_location=CHECKPOINT,
    output_path=OUTPUT_DELTA if "/" in OUTPUT_DELTA else "/Volumes/drone/inference_results",
    config=cfg,
    batch_size=BATCH_SIZE,
    max_files_per_trigger=100,  # back-pressure
)
print("streaming query id:", query.id)

In [ ]:
# Stop the stream when you are done.
# query.stop()